# RuinedFooocus + Face Swap (Colab, one cell)

A fork of [runew0lf/RuinedFooocus](https://github.com/runew0lf/RuinedFooocus) with a **real InsightFace face swap** added, running **inside the notebook** — no `gradio.live`, no public URL.

**Before running:** `Runtime → Change runtime type → T4 GPU`. Leave the *Runtime version* on **Latest**.

Run the single cell below. It clones the fork, installs, starts the app, waits for it, shows the UI, then downloads the models. First run ~15–25 min; later runs ~2 min.

> `inswapper_128.onnx` is InsightFace's *non-commercial research* model. Personal use is fine; commercial use needs a licence.


In [ ]:
#@title Run RuinedFooocus + Face Swap  { display-mode: "form" }
#@markdown Leave this on Latest runtime. Do **not** change the runtime version.
SKIP_CHECKPOINT = False #@param {type:"boolean"}
RESTART_APP     = False #@param {type:"boolean"}

import os, sys, subprocess, time, urllib.request

REPO     = "https://github.com/ahmrafi22/RuinedFooocus.git"
APP_DIR  = "/content/RuinedFooocus"
LOG      = "/content/fooocus.log"
PORT     = 7865
URL      = f"http://127.0.0.1:{PORT}/"


def rule(title=""):
    print("\n" + "=" * 62)
    if title:
        print(title)
        print("=" * 62)


def tail(n=60):
    if os.path.exists(LOG):
        return "\n".join(open(LOG, encoding="utf-8", errors="ignore").read().splitlines()[-n:])
    return "(no log)"


def up():
    try:
        with urllib.request.urlopen(URL, timeout=3) as r:
            return r.status == 200
    except Exception:
        return False


# ---------------------------------------------------------------- 1. machine
rule("1/5  machine")
print("Python:", sys.version.split()[0])
try:
    import torch
    print("torch :", torch.__version__)
except ImportError:
    print("torch : (not installed yet)")

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if smi.returncode == 0:
    print("GPU   :", smi.stdout.strip())
else:
    print("GPU   : NOT FOUND -> Runtime > Change runtime type > T4 GPU")
    print("        (continuing anyway; it will be unusably slow)")

# ---------------------------------------------------------------- 2. code
rule("2/5  code")
if os.path.isdir(APP_DIR):
    print("updating existing checkout...")
    subprocess.run(["git", "-C", APP_DIR, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, APP_DIR], check=True)
os.chdir(APP_DIR)
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

# ---------------------------------------------------------------- 3. start
rule("3/5  starting the app")
# launch.py installs deps, clones ComfyUI, then starts the web UI and never
# returns (webui.py ends in `while True`), so it has to be a background process.
# NOTE: do NOT pass --offline. In launch.py that skips pip entirely, which is
# why onnxruntime/insightface never got installed.

if up() and not RESTART_APP:
    print("already running on port", PORT)
    proc = None
else:
    if up():
        print("restarting...")
        subprocess.run(["pkill", "-f", "launch.py"], check=False)
        time.sleep(3)

    # Colab ships its own CUDA torch; this file stops torchruntime reinstalling it.
    open("freezetorch", "w").close()

    env = dict(os.environ)
    env["TORCH_PLATFORM"] = "cu124"
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    log = open(LOG, "w")
    proc = subprocess.Popen(
        [sys.executable, "launch.py", "--nobrowser",
         "--listen", "127.0.0.1", "--port", str(PORT)],
        stdout=log, stderr=subprocess.STDOUT, env=env, cwd=APP_DIR,
    )
    print("started PID", proc.pid)

# ---------------------------------------------------------------- 4. wait
rule("4/5  installing + starting  (first run ~10-15 min, later runs ~2 min)")
print("live log below. This cell returns only when the UI answers.\n")

deadline = time.time() + 60 * 40
pos = 0
ready = False

while time.time() < deadline:
    if os.path.exists(LOG):
        text = open(LOG, encoding="utf-8", errors="ignore").read()
        if len(text) > pos:
            for line in text[pos:].splitlines():
                line = line.strip()
                # skip tqdm bars, keep everything else
                if not line or ("%" in line and ("|" in line or "it/s" in line or "B/s" in line)):
                    continue
                print("   ", line[:150])
            pos = len(text)

    if proc is not None and proc.poll() is not None:
        print("\nFAILED - the app exited with code", proc.returncode)
        print("-" * 62)
        print(tail(60))
        raise SystemExit("see the log above")

    if up():
        ready = True
        break

    time.sleep(5)

if not ready:
    print("\nFAILED - the app did not come up within 40 minutes")
    print("-" * 62)
    print(tail(60))
    raise SystemExit("see the log above")

print("\nREADY")

# ---------------------------------------------------------------- 5. UI
rule("5/5  UI")
from google.colab import output as colab_output
colab_output.serve_kernel_port_as_iframe(PORT, path="/", height=900, width="100%")

print("Open the **Face Swap** tab: upload a face, tick 'Enable face swap', generate.")
print("If the frame is blank, re-run this cell - the app keeps running.")

# ---------------------------------------------------------------- models
rule("models")
CKPT = os.path.join(APP_DIR, "models/checkpoints/sd_xl_base_1.0_0.9vae.safetensors")
CKPT_URL = ("https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/"
            "resolve/main/sd_xl_base_1.0_0.9vae.safetensors")

if SKIP_CHECKPOINT:
    print("skipping checkpoint download (SKIP_CHECKPOINT is on)")
elif os.path.exists(CKPT) and os.path.getsize(CKPT) > 6e9:
    print("checkpoint already present")
else:
    print("downloading SDXL base 1.0 (~6.9 GB)...")

    def _progress(blocks, block_size, total):
        done = blocks * block_size
        if total > 0 and blocks % 300 == 0:
            print(f"   {done/1e9:5.1f} / {total/1e9:5.1f} GB", end="\r")

    os.makedirs(os.path.dirname(CKPT), exist_ok=True)
    urllib.request.urlretrieve(CKPT_URL, CKPT, _progress)
    print(f"\n   saved ({os.path.getsize(CKPT)/1e9:.1f} GB)")

print("\nface swap models (843 MB, only if not cached):")
sys.path.insert(0, APP_DIR)
import modules.faceswap as faceswap
faceswap.get_swapper()
faceswap.get_analyzer(640)
print(faceswap.status())

rule("all set")
print("The UI above is live. Stop it with:  pkill -f launch.py")